# DQ-11 · geography.csv
Checks: null rate, duplicates, domain values, FK reverse (zip coverage in customers & orders).

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── helpers ───────────────────────────────────────────────────────────────────
issues = []

def flag(label, mask_or_count, df=None, show_sample=True):
    count = int(mask_or_count.sum()) if hasattr(mask_or_count, 'sum') else int(mask_or_count)
    total = len(df) if df is not None else None
    pct   = f' ({count/total*100:.2f}%)' if total else ''
    status = '❌ ISSUE' if count > 0 else '✅ OK'
    print(f'{status}  {label}: {count:,}{pct}')
    if count > 0:
        issues.append(label)
        if show_sample and df is not None and hasattr(mask_or_count, 'sum'):
            sample = df[mask_or_count].head(3)
            print(sample.to_string(index=False))
    return count

def summary():
    print()
    if issues:
        print(f'══ {len(issues)} issue(s) found ══')
        for i in issues: print(f'  • {i}')
    else:
        print('══ All checks passed ══')


In [ ]:
geo    = pd.read_csv('geography.csv')
cust   = pd.read_csv('customers.csv')
orders = pd.read_csv('orders.csv')
print(f'Shape: {geo.shape}')
geo.head(3)

## 1. Null rate

In [ ]:
null_counts = geo.isnull().sum()
print(pd.DataFrame({'null_count': null_counts, 'null_%': (null_counts/len(geo)*100).round(2)}))

## 2. Duplicate zip

In [ ]:
flag('Duplicate zip', geo.duplicated(subset='zip'), geo)

## 3. Domain values: region

In [ ]:
VALID_REGION = {'East','Central','West'}
flag('Invalid region', ~geo['region'].isin(VALID_REGION), geo)

## 4. Reverse FK: customer zips covered by geography

In [ ]:
geo_zips  = set(geo['zip'])
cust_zips = set(cust['zip'])
missing   = cust_zips - geo_zips
flag('Customer zip not in geography', len(missing), show_sample=False)
print(f'  Missing zips (sample): {list(missing)[:10]}')

## 5. Reverse FK: order zips covered by geography

In [ ]:
order_zips  = set(orders['zip'])
missing_ord = order_zips - geo_zips
flag('Order zip not in geography', len(missing_ord), show_sample=False)
print(f'  Missing zips (sample): {list(missing_ord)[:10]}')

## 6. zip ↔ city consistency (1 zip = 1 city?)

In [ ]:
cities_per_zip = geo.groupby('zip')['city'].nunique()
flag('zip maps to multiple cities', (cities_per_zip > 1).sum(), show_sample=False)
print(geo.groupby('zip')['city'].nunique().describe())

## 7. zip ↔ region consistency

In [ ]:
regions_per_zip = geo.groupby('zip')['region'].nunique()
flag('zip maps to multiple regions', (regions_per_zip > 1).sum(), show_sample=False)

## 8. City name consistency across customers & geography

In [ ]:
zip_city = geo.set_index('zip')['city'].to_dict()
cust['expected_city'] = cust['zip'].map(zip_city)
mismatch = cust['city'].notna() & (cust['city'] != cust['expected_city'])
flag('Customer city ≠ geography city for same zip', mismatch, cust)

## Summary

In [ ]:
summary()